# Book Exercises Session 8. Multiple regression with qualitative information

In this workshop you use **dummy (binary) variables** in multiple regression: group differences in the intercept, interactions between a dummy and a continuous variable, tests that a whole set of dummies or interactions is zero, the **Chow test** for equality of two regression functions, and the **linear probability model** (LPM) for a binary dependent variable. The eight exercises follow the mechanics of the computer exercises C2, C3, C4, C6, C7, C8, C9 and C10 in Wooldridge, *Introductory Econometrics*, chapter 7, but with new contexts, new variable names and new data. You write the code yourselves in the empty cells. Work alone or in a group, and ask the TA when you are stuck.




## Toolbox: Python

The functions you need today. `df` is a DataFrame; `m` is a fitted model (`m = smf.ols(...).fit()`).

| Function | What it does | Tiny example |
|---|---|---|
| `df["new"] = ...` | Creates a new column | `df["lwage"] = np.log(df["wage"])` |
| `np.log(x)`, `np.exp(x)` | Natural log and exponential | `100 * (np.exp(b) - 1)` |
| `np.where(cond, a, b)` | `a` where the condition holds, `b` otherwise (builds dummies) | `df["d"] = np.where(df["x"] >= 0.5, 1, 0)` |
| `x.mean()`, `len(df)` | Mean of a column, number of rows | `educ_bar = df["educ"].mean()` |
| `x.value_counts(normalize=True)` | Shares of each value (a fraction for a 0/1 dummy) | `df["eligible"].value_counts(normalize=True)` |
| `pd.crosstab(a, b, normalize="index")` | Two-way table; `normalize="index"` gives row percentages | `pd.crosstab(df["y"], df["y_pred"])` |
| `smf.ols("y ~ x1 + x2", data=df).fit()` | OLS with a formula; the intercept is added automatically | `m = smf.ols("lwage ~ educ + female", data=df).fit()` |
| `I(x**2)` in a formula | A transformed regressor, e.g. a square | `"y ~ exper + I(exper**2)"` |
| `C(var)` in a formula | Turns a categorical column into dummies (first category = base) | `"y ~ C(position)"` |
| `C(var, Treatment("b"))` | Same, with `"b"` as the base group | `"y ~ C(position, Treatment('forward'))"` |
| `a:b` in a formula | Only the interaction (product) $a \times b$ | `"lwage ~ educ + female + female:educ"` |
| `a*b` in a formula | $a + b + a\!:\!b$ (main effects and interaction) | `"sleep ~ male*(totwrk + educ)"` |
| `m.summary()` | Full regression table | `print(m.summary())` |
| `m.params`, `m.bse`, `m.tvalues`, `m.pvalues` | Coefficients, standard errors, $t$ statistics, $p$-values | `m.params["female"]` |
| `m.conf_int(alpha=0.05)` | Confidence intervals for the coefficients | `m.conf_int().loc["female"]` |
| `m.rsquared`, `m.rsquared_adj`, `m.nobs` | $R^2$, adjusted $R^2$, $n$ | `m.rsquared` |
| `m.ssr`, `m.df_resid` | Sum of squared residuals, residual degrees of freedom $n-k-1$ | `m.ssr` |
| `m.fittedvalues` | Fitted values $\hat y_i$ | `m.fittedvalues.min()` |
| `m.f_test("a = 0, b = 0")` | $F$ test of several linear restrictions (use the names in `m.params`) | `m.f_test("I(exper ** 2) = 0, I(tenure ** 2) = 0")` |
| `m.t_test("a + 32*b = 0")` | $t$ test (estimate, se, CI) of one linear combination | `m.t_test("majority + 32*majority:otheroblig = 0")` |
| `m.predict(newdata)` | Predictions for new rows (a DataFrame with the regressors) | `m.predict(pd.DataFrame({"educ": [12], "female": [1]}))` |
| `stats.f.sf(F, q, df)` | $p$-value of an $F$ statistic | `stats.f.sf(2.1, 6, 340)` |
| `stats.f.ppf(1 - a, q, df)` | Critical value of the $F$ distribution | `stats.f.ppf(0.95, 6, 694)` |
| `stats.t.ppf(1 - a/2, df)` | Critical value of the $t$ distribution | `stats.t.ppf(0.975, m.df_resid)` |

*Tip.* Names with spaces inside `m.params` (for example `I(exper ** 2)`) must be written exactly like that inside `f_test` / `t_test`. Print `m.params.index` to see them.

## Toolbox: formulas

**Dummy variable in a level model.** In $y = \beta_0 + \delta_0 d + \beta_1 x + u$, $\delta_0$ is the difference in the expected value of $y$ between the group with $d=1$ and the base group ($d=0$), holding $x$ fixed.

**Dummy variable in a log-level model.** In $\log(y) = \beta_0 + \delta_0 d + \dots$, the approximate percentage difference is $100\,\hat\delta_0\,\%$; the exact one is
$$100\,[\exp(\hat\delta_0) - 1]\,\%.$$

**Several categories.** With $g$ groups use $g-1$ dummies; the left-out group is the **base group**, and each coefficient is the difference to the base group. Including all $g$ dummies *and* an intercept gives perfect collinearity (the *dummy variable trap*). The difference between two non-base groups $j$ and $h$ is $\hat\delta_j - \hat\delta_h$.

**Interaction dummy × continuous.** In $y = \beta_0 + \delta_0 d + \beta_1 x + \delta_1 d\cdot x + u$ the slope of $x$ is $\beta_1$ for $d=0$ and $\beta_1+\delta_1$ for $d=1$. The group difference at a value $x$ is $\delta_0 + \delta_1 x$ (so $\delta_0$ alone is the difference at $x = 0$).

**Centering an interaction.** Replace $d\cdot x$ by $d\cdot(x - c)$. Then the coefficient on $d$ is the group difference at $x = c$ (for example $c = \bar x$), with its own standard error. The other coefficients and the fit do not change.

**$F$ test of $q$ exclusion restrictions** ($r$ = restricted, $ur$ = unrestricted, $k$ slopes in the unrestricted model):
$$F = \frac{(SSR_r - SSR_{ur})/q}{SSR_{ur}/(n-k-1)} \sim F_{q,\,n-k-1}\ \text{under } H_0.$$

**Chow test** for equality of the regression functions of two groups, each model with $k$ slopes plus an intercept ($SSR_P$ = pooled model, $SSR_1$, $SSR_2$ = separate models):
$$F = \frac{SSR_P - (SSR_1 + SSR_2)}{SSR_1 + SSR_2}\cdot\frac{n - 2(k+1)}{k+1} \sim F_{k+1,\ n-2(k+1)}.$$
The same statistic comes from the full sample by adding the group dummy and all its interactions with the regressors and testing that these $k+1$ terms are jointly zero. If only the **slopes** are tested (the dummy itself stays in), $q = k$.

**Linear probability model (LPM).** With $y \in \{0,1\}$, $P(y = 1 \mid x) = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k$. A coefficient $\beta_j$ is the change in the **probability** that $y = 1$ when $x_j$ rises by one unit (× 100 = percentage points). Fitted values can lie below 0 or above 1, and the errors are heteroskedastic.

**Percent correctly predicted.** Set $\tilde y_i = 1$ if $\hat y_i \ge 0.5$ and $\tilde y_i = 0$ otherwise; the overall percent correctly predicted is the share of observations with $\tilde y_i = y_i$. Report it separately for $y=0$ and $y=1$ too.

## Setup

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
sample = pd.read_csv(url + "orders_sample.csv")

wages = pd.read_csv(url + "employee_wages_dk.csv")         # Exercise 1
foot = pd.read_csv(url + "football_salaries.csv")          # Exercise 2
gpa = pd.read_csv(url + "university_gpa.csv")              # Exercise 3
sleep = pd.read_csv(url + "sleep_work_time.csv")           # Exercise 4
serv = pd.read_csv(url + "service_wages_gender.csv")       # Exercise 5
loans = pd.read_csv(url + "business_loans.csv")            # Exercise 6
pens = pd.read_csv(url + "pension_eligibility.csv")        # Exercise 7
nba = pd.read_csv(url + "basketball_players.csv")          # Exercise 8

pd.set_option("display.float_format", "{:.4f}".format)

---
## Exercise 1. The wage gap for employees with an immigrant background (book C7.2)

A trade union wants to know whether employees with a non-Western immigrant background earn less than comparable colleagues. The file `employee_wages_dk.csv` has $n = 935$ full-time private-sector employees in Denmark (**simulated data**).

| Column | Description |
|---|---|
| `wage` | Monthly salary before tax, DKK |
| `educ` | Years of education |
| `exper` | Years of labour-market experience |
| `tenure` | Years with the current employer |
| `married` | 1 if married or cohabiting, 0 otherwise |
| `immigrant` | 1 if non-Western immigrant background, 0 otherwise |
| `north` | 1 if living in North or West Jutland, 0 otherwise |
| `urban` | 1 if living in a city with more than 50,000 inhabitants |

i. Estimate the model
$$\log(wage) = \beta_0 + \beta_1 educ + \beta_2 exper + \beta_3 tenure + \beta_4 married + \beta_5 immigrant + \beta_6 north + \beta_7 urban + u$$
and report the results in the usual form. Holding the other factors fixed, what is the approximate difference in monthly salary between employees with and without an immigrant background? Is this difference statistically significant?

ii. Add the variables $exper^2$ and $tenure^2$ to the equation and show that they are jointly insignificant even at the 20% level.

iii. Extend the original model to allow the return to education to depend on immigrant background, and test whether the return to education does depend on it.

iv. Again starting from the original model, allow wages to differ across four groups: married with immigrant background, married without, single with immigrant background and single without. What is the estimated wage differential between married employees with and without an immigrant background?

In [ ]:
# Hint: create lwage = np.log(wage), then smf.ols("lwage ~ educ + exper + ... + urban", data=wages).fit()
# Hint: approximate % difference = 100 * coefficient; exact = 100 * (np.exp(b) - 1). Look at the t statistic / p-value.

**Your answer:**



In [ ]:
# Hint: add I(exper**2) + I(tenure**2) to the formula, then m.f_test("I(exper ** 2) = 0, I(tenure ** 2) = 0").
# Hint: or by hand: F = ((SSR_r - SSR_ur)/q) / (SSR_ur/df_resid); compare with stats.f.ppf(0.80, q, df).

**Your answer:**



In [ ]:
# Hint: add the interaction immigrant:educ to the original model and look at its t statistic.

**Your answer:**



In [ ]:
# Hint: build three of the four group dummies with products, e.g. married * immigrant, married * (1 - immigrant), ...
# Hint: leave out one group (base). The married-immigrant vs married-non-immigrant gap is the difference of two coefficients;
#       m.t_test("marr_imm - marr_nonimm = 0") gives its standard error.

**Your answer:**



---
## Exercise 2. Football salaries and playing position (book C7.3)

A sports economist studies whether professional footballers in a European first division are paid differently by playing position once their experience and performance are taken into account. The file `football_salaries.csv` has $n = 353$ players (**simulated data**). Annual statistics are per season averages over the career.

| Column | Description |
|---|---|
| `salary` | Annual salary, EUR |
| `years` | Seasons as a professional |
| `gamesyr` | Games played per season |
| `goalsyr`, `assistsyr` | Goals and assists per season |
| `keypassyr`, `tacklesyr` | Key passes and tackles per season |
| `passpct` | Pass completion, % |
| `natteam` | % of seasons called up to the national team |
| `position` | Text label of the position |
| `goalkeeper`, `centreback`, `fullback`, `defmid`, `attmid`, `winger` | Position dummies; **forwards** (strikers) are the base group |

Consider the model
$$\log(salary) = \beta_0 + \beta_1 years + \beta_2 gamesyr + \beta_3 goalsyr + \beta_4 assistsyr + \beta_5 keypassyr + \beta_6 tacklesyr + \beta_7 passpct + \beta_8 natteam$$
$$+ \beta_9 goalkeeper + \beta_{10} centreback + \beta_{11} fullback + \beta_{12} defmid + \beta_{13} attmid + \beta_{14} winger + u.$$

i. State the null hypothesis that, controlling for the other factors, goalkeepers and forwards earn, on average, the same amount. Test this hypothesis using the data and comment on the size of the estimated salary differential.

ii. State and test the null hypothesis that there is no difference in average salary across positions, once the other factors have been controlled for.

iii. Are the results from parts (i) and (ii) consistent? If not, explain what is happening.

In [ ]:
# Hint: lsalary = np.log(salary); estimate the full model with the six position dummies (forwards = base group).
# Hint: H0: beta_goalkeeper = 0. Use the t statistic; the exact % gap is 100*(np.exp(b) - 1).

**Your answer:**



In [ ]:
# Hint: H0: all six position coefficients are 0. Use m.f_test("goalkeeper = 0, centreback = 0, ...")
#       or the restricted/unrestricted SSR formula from the toolbox (q = 6).

**Your answer:**



**Your answer:**



---
## Exercise 3. University grades, elite athletes and gender (book C7.4)

A university wants to know how the grade point average of its first-year students relates to their background, and in particular whether students in the **elite-sport programme** do worse or better than other students. The file `university_gpa.csv` has $n = 4{,}137$ students (**simulated data**).

| Column | Description |
|---|---|
| `colgpa` | Grade point average after the first year (scale 0–4) |
| `hsize` | Size of the student's high-school graduating class, in hundreds |
| `hsperc` | Rank in the high-school class, percentile from the **top** (low = better) |
| `admtest` | Score on the national admission test (500–1600) |
| `female` | 1 if female |
| `athlete` | 1 if the student is in the elite-sport programme |

Consider the model
$$colgpa = \beta_0 + \beta_1 hsize + \beta_2 hsize^2 + \beta_3 hsperc + \beta_4 admtest + \beta_5 female + \beta_6 athlete + u.$$

i. What are your expectations for the coefficients in this equation? Which ones are you unsure about?

ii. Estimate the equation and report the results in the usual form. What is the estimated GPA differential between athletes and nonathletes? Is it statistically significant?

iii. Drop `admtest` from the model and reestimate the equation. Now what is the estimated effect of being an athlete? Discuss why the estimate is different from that obtained in part (ii).

iv. In the model from part (i), allow the effect of being an athlete to differ by gender and test the null hypothesis that there is no *ceteris paribus* difference between women athletes and women nonathletes.

v. Does the effect of `admtest` on `colgpa` differ by gender? Justify your answer.

**Your answer:**



In [ ]:
# Hint: smf.ols("colgpa ~ hsize + I(hsize**2) + hsperc + admtest + female + athlete", data=gpa).fit()

**Your answer:**



In [ ]:
# Hint: drop admtest and reestimate. Then compare gpa.groupby("athlete")["admtest"].mean() (omitted variable bias).

**Your answer:**



In [ ]:
# Hint: add female:athlete. Women athletes vs women nonathletes = athlete + female:athlete -> m.t_test("athlete + female:athlete = 0").
# Hint: alternative: group dummies with female nonathletes as the base group; then test one coefficient.

**Your answer:**



In [ ]:
# Hint: add the interaction female:admtest to the model of part (ii) and test it.

**Your answer:**



---
## Exercise 4. Sleep and work time of men and women: the Chow test (book C7.6)

A public-health researcher uses a time-use survey to study the trade-off between work and sleep. The file `sleep_work_time.csv` has $n = 706$ working-age adults (**simulated data**).

| Column | Description |
|---|---|
| `sleep` | Minutes of night sleep per week |
| `totwrk` | Minutes of paid work per week |
| `educ` | Years of education |
| `age` | Age in years |
| `yngkid` | 1 if there is a child under 3 in the household |
| `male` | 1 if male |

The model is
$$sleep = \beta_0 + \beta_1 totwrk + \beta_2 educ + \beta_3 age + \beta_4 age^2 + \beta_5 yngkid + u.$$

i. Estimate this equation separately for men and women and report the results in the usual form. Are there notable differences in the two estimated equations?

ii. Compute the Chow test for equality of the parameters in the sleep equation for men and women. Use the form of the test that adds `male` and the interaction terms `male*totwrk`, …, `male*yngkid` and uses the full set of observations. What are the relevant $df$ for the test? Should you reject the null at the 5% level?

iii. Now allow for a different intercept for males and females and determine whether the interaction terms involving `male` are jointly significant.

iv. Given the results from parts (ii) and (iii), what would be your final model?

In [ ]:
# Hint: estimate the same formula on sleep[sleep["male"] == 1] and on sleep[sleep["male"] == 0].
# Hint: put the two sets of params / bse side by side in a DataFrame to compare.

**Your answer:**



In [ ]:
# Hint: "sleep ~ male*(totwrk + educ + age + I(age**2) + yngkid)" adds male and all interactions.
# Hint: test male and all male:... terms jointly with f_test (k + 1 = 6 restrictions), or use the Chow formula with
#       the SSR of the pooled model and of the two separate models.

**Your answer:**



In [ ]:
# Hint: keep male in the model and test only the five male:... interaction terms (q = 5).

**Your answer:**



In [ ]:
# Hint: estimate the model you choose and show its coefficients.

**Your answer:**



---
## Exercise 5. A gender wage gap that depends on education: centering an interaction (book C7.7)

An analyst for a service-sector employers' organisation studies the hourly pay of shop and hospitality workers. The file `service_wages_gender.csv` has $n = 526$ workers (**simulated data**).

| Column | Description |
|---|---|
| `wage` | Hourly wage, DKK |
| `female` | 1 if female |
| `educ` | Years of education |
| `exper` | Years of potential experience |
| `tenure` | Years with the current employer |

The model (the analogue of equation (7.18) in the book) is
$$\log(wage) = \beta_0 + \beta_1 female + \beta_2 educ + \beta_3 female\cdot educ + \beta_4 exper + \beta_5 exper^2 + \beta_6 tenure + \beta_7 tenure^2 + u.$$

i. Estimate the equation. Use it to obtain the estimated gender differential when `educ` equals its sample mean $\overline{educ}$ (about 13.2 years). Compare this with the estimated differential when $educ = 0$.

ii. Run the regression used to obtain the equation, but with $female\cdot(educ - \overline{educ})$ replacing $female\cdot educ$. How do you interpret the coefficient on `female` now?

iii. Is the coefficient on `female` in part (ii) statistically significant? Compare this with the coefficient and significance of `female` in part (i) and comment.

In [ ]:
# Hint: estimate the model with female:educ. The differential at a given educ is b_female + b_female:educ * educ.

**Your answer:**



In [ ]:
# Hint: make a column educ_c = educ - educ mean, and use female:educ_c instead of female:educ.

**Your answer:**



In [ ]:
# Hint: compare params, bse and pvalues of female in the two models.

**Your answer:**



---
## Exercise 6. Small-business loan approvals and the owner's group (book C7.8)

A regulator investigates whether banks treat small-business loan applications from owners of the majority ethnic group and from minority owners differently. The file `business_loans.csv` has $n = 1{,}989$ applications (**simulated data**).

| Column | Description |
|---|---|
| `approve` | 1 if the loan was approved |
| `majority` | 1 if the owner belongs to the majority group, 0 if minority |
| `payratio` | Loan repayments as % of the business's income |
| `otheroblig` | Other debt obligations as % of income |
| `loanprc` | Loan amount / value of the collateral |
| `unem` | Unemployment rate in the business's sector, % |
| `male`, `married` | 1 if the owner is male / married |
| `dep` | Number of dependants of the owner |
| `degree` | 1 if the owner has a university degree |
| `cosign` | 1 if the loan has a co-signer |
| `goodcredit` | 1 if the owner has a good credit history |
| `pubrec` | 1 if there is a public record of a past default |
| `late1`, `late2` | 1 if one / two or more late payments on previous loans |
| `highrisk` | 1 if the business is in a sector the bank classifies as high-risk |

The simplest model is the linear probability model $approve = \beta_0 + \beta_1 majority + \text{other factors}$.

i. If there is discrimination against minority owners, and the appropriate factors have been controlled for, what is the sign of $\beta_1$?

ii. Regress `approve` on `majority` and report the results in the usual form. Interpret the coefficient on `majority`. Is it statistically significant? Is it practically large?

iii. As controls, add the variables `payratio`, `otheroblig`, `loanprc`, `unem`, `male`, `married`, `dep`, `degree`, `cosign`, `goodcredit`, `pubrec`, `late1`, `late2` and `highrisk`. What happens to the coefficient on `majority`? Is there still evidence of discrimination against minority owners?

iv. Now allow the effect of the owner's group to interact with the variable measuring other obligations as a percentage of income (`otheroblig`). Is the interaction term significant?

v. Using the model from part (iv), what is the effect of belonging to the majority group on the probability of approval when $otheroblig = 32$, which is roughly the mean value in the sample? Obtain a 95% confidence interval for this effect.

**Your answer:**



In [ ]:
# Hint: smf.ols("approve ~ majority", data=loans).fit(). Compare with the approval rate in each group (groupby + mean).

**Your answer:**



In [ ]:
# Hint: put the controls in a string and add it to the formula: "approve ~ majority + " + controls.

**Your answer:**



In [ ]:
# Hint: add majority:otheroblig to the model of part (iii) and look at its t statistic.

**Your answer:**



In [ ]:
# Hint: effect = b_majority + 32 * b_majority:otheroblig. Use m.t_test("majority + 32*majority:otheroblig = 0")
#       or re-centre: oblig_c = otheroblig - 32, interact majority with oblig_c, and read conf_int() for majority.

**Your answer:**



---
## Exercise 7. Eligibility for an employer pension scheme (book C7.9)

A pensions think tank studies which employees have access to a voluntary employer-sponsored pension scheme (with an employer contribution). The file `pension_eligibility.csv` has $n = 9{,}275$ households (**simulated data**).

| Column | Description |
|---|---|
| `eligible` | 1 if the household is eligible for an employer pension scheme |
| `inc` | Annual household income, thousand EUR |
| `age` | Age of the main earner, years |
| `male` | 1 if the main earner is a single male (the comparison is with all other households) |
| `privpen` | 1 if the household has a private (individual) pension savings account |

i. What fraction of the households in the sample are eligible for an employer pension scheme?

ii. Estimate a linear probability model explaining eligibility in terms of income, age and gender. Include income and age in quadratic form, and report the results in the usual form.

iii. Would you say that eligibility is independent of income and age? What about gender? Explain.

iv. Obtain the fitted values from the linear probability model estimated in part (ii). Are any fitted values negative or greater than one?

v. Using the fitted values $\widehat{eligible}_i$ from part (iv), define $\widetilde{eligible}_i = 1$ if $\widehat{eligible}_i \ge 0.5$ and $\widetilde{eligible}_i = 0$ if $\widehat{eligible}_i < 0.5$. Out of the 9,275 households, how many are predicted to be eligible?

vi. For the households not eligible, what percent of these are predicted not to be eligible using $\widetilde{eligible}$? For the households eligible, what percent are predicted to be eligible? (Tabulate $eligible$ against $\widetilde{eligible}$.)

vii. The overall percent correctly predicted is the share of households with $\widetilde{eligible}_i = eligible_i$. Compute it. Do you think this is a complete description of how well the model does, given your answers in part (vi)?

viii. Add the variable `privpen` as an explanatory variable to the linear probability model. Other things equal, if a household has a private pension account, how much higher is the estimated probability that it is eligible for an employer scheme? Is it statistically different from zero at the 10% level?

In [ ]:
# Hint: the mean of a 0/1 variable is the fraction of ones (or value_counts(normalize=True)).

**Your answer:**



In [ ]:
# Hint: smf.ols("eligible ~ inc + I(inc**2) + age + I(age**2) + male", data=pens).fit()

**Your answer:**



In [ ]:
# Hint: test inc and I(inc ** 2) jointly with f_test; the same for age; look at the t statistic of male.
# Hint: turning point of a quadratic b1*x + b2*x^2 is -b1 / (2*b2).

**Your answer:**



In [ ]:
# Hint: m.fittedvalues; count how many are < 0 and > 1 with (condition).sum().

**Your answer:**



In [ ]:
# Hint: np.where(fitted >= 0.5, 1, 0), then sum.

**Your answer:**



In [ ]:
# Hint: pd.crosstab(actual, predicted) and the same with normalize="index" (row percentages).

**Your answer:**



In [ ]:
# Hint: mean of (predicted == actual). Compare with the share you would get by always predicting the most common outcome.

**Your answer:**



In [ ]:
# Hint: add privpen; its coefficient is the change in probability. Compare the p-value with 0.10 (or use conf_int(alpha=0.10)).

**Your answer:**



---
## Exercise 8. Points per game and position in a basketball league (book C7.10)

A scout for a European basketball club studies what explains players' scoring. The file `basketball_players.csv` has $n = 269$ players (**simulated data**).

| Column | Description |
|---|---|
| `points` | Points per game (career average) |
| `assists` | Assists per game (career average) |
| `exper` | Years as a professional |
| `age` | Age in years |
| `position` | Text label: guard, forward or center |
| `guard`, `forward`, `center` | Position dummies |
| `marr` | 1 if married |

i. Estimate a linear regression model relating points per game to experience in the league and position (guard, forward or center). Include experience in quadratic form and use centers as the base group. Report the results in the usual form.

ii. Why do you not include all three position dummy variables in part (i)?

iii. Holding experience fixed, does a guard score more than a center? How much more? Is the difference statistically significant?

iv. Now, add marital status to the equation. Holding position and experience fixed, are married players more productive (based on points per game)?

v. Add interactions of marital status with both experience variables. In this expanded model, is there strong evidence that marital status affects points per game?

vi. Estimate the model from part (iv) but use assists per game as the dependent variable. Are there any notable differences from part (iv)? Discuss.

In [ ]:
# Hint: include guard and forward (center = base) and exper + I(exper**2).
# Hint: you can also use C(position, Treatment('center')) and get the same numbers.

**Your answer:**



**Your answer:**



In [ ]:
# Hint: the coefficient on guard is the guard - center difference. H1: beta_guard > 0 (one-sided: p/2 if the sign is right).

**Your answer:**



In [ ]:
# Hint: add marr and look at its coefficient and t statistic.

**Your answer:**



In [ ]:
# Hint: add marr:exper and marr:I(exper**2). Marital status "matters" if marr and both interactions are jointly significant -> f_test with 3 restrictions.

**Your answer:**



In [ ]:
# Hint: same right-hand side as part (iv), with assists as y. Compare the two sets of coefficients and p-values.

**Your answer:**



---
## Wrap-up

* A dummy shifts the **intercept**; a dummy × continuous interaction shifts the **slope**.
* Always know your **base group**: every dummy coefficient is a difference *to the base group*.
* Individual $t$ tests and a joint $F$ test answer different questions and can disagree (Exercise 2).
* The Chow test asks whether *anything* differs between two groups; testing only the interactions asks whether the *slopes* differ (Exercise 4).
* Centering ($x - \bar x$) in an interaction makes the dummy coefficient the group gap at a meaningful value (Exercises 5 and 6).
* In an LPM coefficients are changes in **probability**; check the fitted values and report the percent correctly predicted for each outcome separately (Exercise 7).